In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np

print("Versión de TensorFlow:", tf.__version__)

Versión de TensorFlow: 2.20.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import tarfile
import os

drive_file_path = '/content/drive/MyDrive/cifar-10-python.tar.gz'

print("Descomprimiendo dataset desde Google Drive...")

try:
    with tarfile.open(drive_file_path, 'r:gz') as tar:
        tar.extractall(path='./')
    print("¡Dataset descomprimido exitosamente en Colab!")
except FileNotFoundError:
    print(f" Error: No se encontró el archivo en la ruta '{drive_file_path}'.")
    print("Verifica si el nombre o la carpeta en Drive es diferente.")

Descomprimiendo dataset desde Google Drive...


/tmp/ipykernel_2264/4286233913.py:14: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path='./')


¡Dataset descomprimido exitosamente en Colab!


In [ ]:

model = models.Sequential([

    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])


model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 30, 30, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 15, 15, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 13, 13, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 6, 6, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 4, 4, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        65,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 122,570 (478.79 KB)

 Trainable params: 122,570 (478.79 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import pickle

def load_cifar_batch(file):
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict

def load_cifar10_data(path):
    x_train_list, y_train_list = [], []
    for i in range(1, 6):
        batch = load_cifar_batch(os.path.join(path, f'data_batch_{i}'))
        x_train_list.append(batch[b'data'])
        y_train_list.extend(batch[b'labels'])

    x_train = np.concatenate(x_train_list)
    y_train = np.array(y_train_list)

    test_batch = load_cifar_batch(os.path.join(path, 'test_batch'))
    x_test = test_batch[b'data']
    y_test = np.array(test_batch[b'labels'])

    x_train = x_train.reshape((len(x_train), 3, 32, 32)).transpose(0, 2, 3, 1).astype('float32') / 255.0
    x_test = x_test.reshape((len(x_test), 3, 32, 32)).transpose(0, 2, 3, 1).astype('float32') / 255.0

    return x_train, y_train, x_test, y_test

data_path = './cifar-10-batches-py'
x_train, y_train, x_test, y_test = load_cifar10_data(data_path)

print("Shape of x_train:", x_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of x_test:", x_test.shape)
print("Shape of y_test:", y_test.shape)

history = model.fit(
    x_train, y_train,
    epochs=15,
    batch_size=64,
    validation_data=(x_test, y_test)
)

Shape of x_train: (50000, 32, 32, 3)
Shape of y_train: (50000,)
Shape of x_test: (10000, 32, 32, 3)
Shape of y_test: (10000,)
Epoch 1/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 77s 96ms/step - accuracy: 0.3831 - loss: 1.6689 - val_accuracy: 0.5079 - val_loss: 1.3522
Epoch 2/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 75s 86ms/step - accuracy: 0.5304 - loss: 1.3112 - val_accuracy: 0.5786 - val_loss: 1.1802
Epoch 3/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 70s 89ms/step - accuracy: 0.5762 - loss: 1.1896 - val_accuracy: 0.6148 - val_loss: 1.0807
Epoch 4/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 69s 89ms/step - accuracy: 0.6185 - loss: 1.0811 - val_accuracy: 0.6521 - val_loss: 0.9846
Epoch 5/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 84s 91ms/step - accuracy: 0.6481 - loss: 1.0015 - val_accuracy: 0.6604 - val_loss: 0.9558
Epoch 6/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 69s 89ms/step - accuracy: 0.6675 - loss: 0.9418 - val_accuracy: 0.6627 - val_loss: 0.9556
Epoch 7/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 69s 88ms/step - accuracy: 0.6860 - loss: 0.8969 - val_

In [ ]:

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print(f"\n Precisión final en datos de prueba: {test_acc * 100:.2f}%")

model.save('cifar10_model.keras')
print(" Modelo guardado como 'cifar10_model.keras'")

from google.colab import files
files.download('cifar10_model.keras')

313/313 - 4s - 12ms/step - accuracy: 0.7222 - loss: 0.8544

🎯 Precisión final en datos de prueba: 72.22%
✅ Modelo guardado como 'cifar10_model.keras'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>